In [ ]:
import pandas as pd


with fs.open("s3://lab/art_net_dec.parquet") as f : 
    dfa = pd.read_parquet(f)

with fs.open("s3://lab/conf_net_dec.parquet") as f : 
    dfc = pd.read_parquet(f)

In [8]:
dfa["authors"] = dfa["authors"].apply(lambda lst: [d["name"] for d in lst])
dfc["authors"] = dfc["authors"].apply(lambda lst: [d["name"] for d in lst])

In [16]:
nla = dfa["authors"].explode().str.split(' ').str[0]
nlc = dfc["authors"].explode().str.split(' ').str[0]

In [19]:
nl = pd.concat([nla,nlc]).drop_duplicates()

In [49]:
nl = nl[~nl.str.contains(r"^[A-Za-z]\.$", na=False)]
nl = nl[~nl.str.contains(r"^[A-Za-z]\.-[A-Za-z]\.$", na=False)]
nl = nl[~nl.str.contains(r"^[A-Za-z]-[A-Za-z]\.$", na=False)]
nl = nl[~nl.str.contains(r"^[A-Za-z][A-Za-z]\.$", na=False)]
nl = nl[~nl.str.contains(r"^[A-Za-z][A-Za-z][A-Za-z]\.$", na=False)]

In [51]:
nl.to_csv("s3://lab/mem/full_name_list.csv")

In [52]:
nl

0                     Kai
1                 Richard
1                  Stefan
2                 Herbert
3                  Sophie
                ...      
3749381          Siwaphon
3749388         Tasi-Ying
3749413           XiangXu
3749415    Pradhyumansinh
3749454              Chah
Name: authors, Length: 313632, dtype: object

In [53]:
!pip install gender-guesser

In [71]:
import gender_guesser.detector as gg

d = gg.Detector()
ng_dict = pd.DataFrame()
ng_dict["name"] = nl
ng_dict["gender"] = nl.apply(lambda x: d.get_gender(x))
ng_dict

,name,gender
0,Kai,male
1,Richard,male
1,Stefan,male
2,Herbert,male
3,Sophie,female
...,...,...
3749381,Siwaphon,unknown
3749388,Tasi-Ying,unknown
3749413,XiangXu,unknown
3749415,Pradhyumansinh,unknown


In [72]:
ng_dict[ng_dict["gender"]=="unknown"]

,name,gender
4,NaN,unknown
27,Hans-Jürgen,unknown
42,Nicolaos,unknown
87,Brikenrikena,unknown
96,Catalin,unknown
...,...,...
3749381,Siwaphon,unknown
3749388,Tasi-Ying,unknown
3749413,XiangXu,unknown
3749415,Pradhyumansinh,unknown
